In [ ]:
# === Imports ===
# Standard library
import gc
import json
import os
from pathlib import Path
from typing import Dict, List, Optional

# Third-party
import numpy as np
import pandas as pd
import torch
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

In [ ]:
login()  # will prompt for your token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_to_run = "Qwen/Qwen3-Embedding-8B"

In [ ]:
# NOTE: Only needed for the Colab / Google-Drive version.
# For local runs, install dependencies from requirements.txt instead.
# !pip install -q --upgrade \
#   transformers==4.57.6 \
#   sentence-transformers==5.2.3

In [ ]:
# GDrive
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
def _exists(p):
    return p and os.path.exists(p)

def resolve_ocr_dir() -> str:
    env_p = os.environ.get("AMR")
    if _exists(env_p):
        return env_p

    candidates = [
        "/content/drive/MyDrive/AMR/",
        "/content/drive/My Drive/AMR/",
    ]
    for p in candidates:
        if _exists(p):
            return p

    share_root = "/content/drive/Shareddrives"
    if _exists(share_root):
        for team in os.listdir(share_root):
            cand = os.path.join(share_root, team, "AMR")
            if _exists(cand):
                return cand

    root = "/content/drive"
    max_depth = 3
    for cur, dirs, _files in os.walk(root):
        depth = cur.count(os.sep) - root.count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(cur) == "AMR":
            return cur

    raise FileNotFoundError(
        "OCR folder not found.\n"
        "Make sure you added a *shortcut* of the shared 'AMR' folder to 'My Drive', "
        "or set AMR to the exact path."

    )

# AMR_DIR = resolve_ocr_dir()
# print("Using OCR data at:", AMR_DIR)

# if not os.path.exists("/content/AMR"):
#     os.symlink(AMR_DIR, "/content/AMR")
# print("Stable path:", "/content/AMR")

AMR_DIR = "."

Mounted at /content/drive
Using OCR data at: /content/drive/MyDrive/AMR/
Stable path: /content/AMR


# WMT24pp

## Setup

In [ ]:
class ModelConfig:
    """Model-specific encoding configurations."""

    KNOWN_CONFIGS = {
        'intfloat/multilingual-e5-base': {'prefix': 'query: ', 'suffix': ''},
        'intfloat/multilingual-e5-large': {'prefix': 'query: ', 'suffix': ''},
        'intfloat/multilingual-e5-large-instruct': {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery: ', 'suffix' : ''},
        'Alibaba-NLP/gte-multilingual-base': {'prefix': '', 'suffix': ''},
        'nomic-ai/nomic-embed-text-v2-moe': {'prefix': 'search_query:', 'suffix': ''},
        'sentence-transformers/paraphrase-multilingual-mpnet-base-v2': {'prefix': '', 'suffix': ''},
        'sentence-transformers/LaBSE': {'prefix': '', 'suffix': ''},
        'BAAI/bge-m3': {'prefix': '', 'suffix': ''},
        'jinaai/jina-embeddings-v3': {'prefix': '', 'suffix': ''},
        'jinaai/jina-embeddings-v5-text-small': {'prefix': '', 'suffix': ''},
        'jinaai/jina-embeddings-v5-text-nano': {'prefix': '', 'suffix': ''},
        'google/embeddinggemma-300m' : {'prefix': 'task: sentence similarity | query:', 'suffix': ''},
        "Qwen/Qwen3-Embedding-0.6B": {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery:', 'suffix' : ''},
        "Qwen/Qwen3-Embedding-4B": {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery:', 'suffix' : ''},
        "Qwen/Qwen3-Embedding-8B": {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery:', 'suffix' : ''},
        'default': {'prefix': '', 'suffix': ''}
    }

    @classmethod
    def get_config(cls, model_name: str, prefix: str = None, suffix: str = None) -> Dict:
        if prefix is not None or suffix is not None:
            return {'prefix': prefix or '', 'suffix': suffix or ''}
        return cls.KNOWN_CONFIGS.get(model_name, cls.KNOWN_CONFIGS['default']).copy()

class WMT24PPEmbeddingGenerator:
    """Generate embeddings for WMT24PP dataset JSON structure."""

    TRANSFORMATIONS = [
        'polarity_negation',
        'role_swap',
        'antonym_replacement',
        'hypernym_substitution'
    ]

    def __init__(self, model_name: str, csv_path: str, prefix: str = None,
                 suffix: str = None, device: str = None, batch_size: int = 16, trust_remote_code=False, jina_v3=False):
        """
        Initialize generator.

        Args:
            model_name: HuggingFace model ID
            csv_path: Path to WMT24PP CSV
            prefix: Optional text prefix for encoding
            suffix: Optional text suffix for encoding
            device: 'cuda' or 'cpu' (auto-detect if None)
            batch_size: Batch size for encoding
        """
        self.csv_path = Path(csv_path)
        self.batch_size = batch_size

        config = ModelConfig.get_config(model_name, prefix, suffix)
        self.prefix = config['prefix']
        self.suffix = config['suffix']
        self.model_name = model_name

        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device

        print(f"\n{'='*80}")
        print(f"Initializing WMT24PP Embedding Generator")
        print(f"{'='*80}")
        print(f"Model: {model_name}")
        print(f"Device: {device}")
        if self.prefix or self.suffix:
            print(f"Prefix: '{self.prefix}'")
            print(f"Suffix: '{self.suffix}'")

        print(f"\nLoading model...")
        self.model = SentenceTransformer(model_name, device=device, trust_remote_code=trust_remote_code)
        print(f" Model loaded")

        if jina_v3:
            self.model[0].default_task = 'text-matching' # if you try another adapter, also add it to config
            print(f" Jina Embeddings V3 was set to the text-matching adapter")

        print(f"\nLoading CSV: {self.csv_path}")
        try:
            self.df = pd.read_csv(self.csv_path, encoding="utf-16")
        except:
            self.df = pd.read_csv(self.csv_path,encoding="utf-16")
        print(f"  ✓ Loaded {len(self.df)} rows")

        self.df = self.df[self.df["is_bad_source"] == False].copy() # filter df when is_bad_source == False
        self.available_languages = self._discover_languages()
        print(f"  ✓ Found {len(self.available_languages)} languages")
        print(f"     First 10: {', '.join(self.available_languages[:10])}")

    def _discover_languages(self) -> List[str]:
        """Discover language codes (en_EN, de_DE format)."""
        languages = []
        for col in self.df.columns:
            if len(col) == 5 and col[2] == '_' and col[:2].isalpha() and col[3:].isalpha():
                languages.append(col)
        return sorted(languages)

    def _encode(self, texts: List[str]) -> np.ndarray:
        """Encode texts with model-specific prefix/suffix."""
        if self.prefix or self.suffix:
            texts = [f"{self.prefix}{text}{self.suffix}" for text in texts]

        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        return embeddings

    def generate(self, output_dir: str,languages: Optional[List[str]] = None) -> Path:
        """
        Generate embeddings in clean JSON structure.

        Structure: {index: {language: {original}, foil_type: {foil}}}
        Languages and foil types are at the same level.

        Args:
            output_dir: Output directory
            languages: Specific languages (None = all)

        Returns:
            Path to output JSON file
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        model_slug = self.model_name.replace('/', '_').replace('-', '_')
        output_file = output_dir / f"wmt24pp_{model_slug}.json"

        if languages is None:
            languages = self.available_languages
            print(f"\n  Using ALL {len(languages)} languages")
        else:
            invalid = set(languages) - set(self.available_languages)
            if invalid:
                print(f"  Warning: Languages not in CSV: {invalid}")
            languages = [lang for lang in languages if lang in self.available_languages]
            print(f"\n  Using {len(languages)} specified languages")

        if not languages:
            raise ValueError("No valid languages found!")

        print(f"\n{'='*80}")
        print(f"Generating Embeddings")
        print(f"{'='*80}")
        print(f"Output: {output_file}")
        print(f"Languages: {len(languages)}")

        data = {}

        print(f"\n1. Collecting successful foils...")
        successful_indices_per_foil = {}
        for foil_type in self.TRANSFORMATIONS:
            status_col = f'foil_{foil_type}_status'
            foil_col = f'foil_{foil_type}_text'

            if status_col in self.df.columns and foil_col in self.df.columns:
                mask = (self.df[status_col] == 'success') & self.df[foil_col].notna()
                successful_indices_per_foil[foil_type] = set(self.df[mask].index.tolist())
                print(f"   {foil_type:30s}: {len(successful_indices_per_foil[foil_type]):4d} successful")

        all_successful_indices = set()
        for indices in successful_indices_per_foil.values():
            all_successful_indices.update(indices)
        all_successful_indices = sorted(all_successful_indices)

        print(f"\n   Total rows with successful foils: {len(all_successful_indices)}")

        for idx in all_successful_indices:
            data[str(idx)] = {}

        print(f"\n2. Encoding originals...")
        for language in tqdm(languages, desc="Languages"):
            valid_rows = self.df.loc[all_successful_indices]
            valid_rows = valid_rows[valid_rows[language].notna()]

            if len(valid_rows) == 0:
                continue

            texts = valid_rows[language].tolist()
            indices = valid_rows.index.tolist()

            embeddings = self._encode(texts)

            for idx, text, emb in zip(indices, texts, embeddings):
                data[str(idx)][language] = {
                    'original_text': text,
                    'embedding': emb.tolist()
                }

        print(f"  Encoded originals for {len(languages)} languages")

        print(f"\n3. Encoding foils...")
        for foil_type in tqdm(self.TRANSFORMATIONS, desc="Foil types"):
            foil_col = f'foil_{foil_type}_text'

            if foil_col not in self.df.columns:
                continue

            indices = sorted(successful_indices_per_foil[foil_type])
            valid_rows = self.df.loc[indices]
            valid_rows = valid_rows[valid_rows[foil_col].notna()]

            if len(valid_rows) == 0:
                continue

            texts = valid_rows[foil_col].tolist()
            indices = valid_rows.index.tolist()

            embeddings = self._encode(texts)

            for idx, text, emb in zip(indices, texts, embeddings):
                idx_str = str(idx)
                if idx_str in data:
                    data[idx_str][foil_type] = {
                        'text': text,
                        'embedding': emb.tolist()
                    }

        print(f" Encoded foils")

        result = {
            'metadata': {
                'model': self.model_name,
                'languages': languages,
                'foil_types': self.TRANSFORMATIONS,
                'total_indices': len(data),
                'encoding_config': {
                    'prefix': self.prefix,
                    'suffix': self.suffix,
                    'device': self.device,
                    'batch_size': self.batch_size
                }
            },
            'data': data
        }

        print(f"\n4. Writing to JSON...")
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)

        file_size_mb = output_file.stat().st_size / 1024 / 1024

        print(f"\n{'='*80}")
        print(f"✓ Generated embeddings for {len(data)} indices")
        print(f"✓ Saved to: {output_file}")
        print(f"✓ Size: {file_size_mb:.1f} MB")
        print(f"{'='*80}")

        # print(f"\n Sample structure:")
        # sample_idx = str(all_successful_indices[0])
        # print(f"\ndata['{sample_idx}'] keys:")
        # sample_keys = list(data[sample_idx].keys())
        # lang_keys = [k for k in sample_keys if len(k) == 5 and k[2] == '_']
        # foil_keys = [k for k in sample_keys if k in self.TRANSFORMATIONS]
        # print(f"  Languages ({len(lang_keys)}): {lang_keys[:5]}{'...' if len(lang_keys) > 5 else ''}")
        # print(f"  Foils ({len(foil_keys)}): {foil_keys}")

        return output_file

## RUN

In [ ]:
models_to_run = [
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    # "intfloat/multilingual-e5-large-instruct",
    # "Qwen/Qwen3-Embedding-0.6B",
    # "Qwen/Qwen3-Embedding-4B",
    # "Qwen/Qwen3-Embedding-8B",
    # "Alibaba-NLP/gte-multilingual-base",
    # "nomic-ai/nomic-embed-text-v2-moe",
    # "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    # "sentence-transformers/LaBSE",
    # "BAAI/bge-m3",
    # "jinaai/jina-embeddings-v3",
    # "jinaai/jina-embeddings-v5-text-small",
    # "jinaai/jina-embeddings-v5-text-nano",
    # "google/embeddinggemma-300m",
    # "ibm-granite/granite-embedding-107m-multilingual",
    # "ibm-granite/granite-embedding-278m-multilingual"
]
# models_to_run = [model_to_run]

# model_name = model_to_run

# g_drive_csv_path = '/content/AMR/wt24_with_foils/wmt24pp_all_foils.csv'
# g_drive_output_dir = '/content/AMR/embeddings'

csv_path = "datasets/alee_mt61.csv"
output_dir = 'intermediate_outputs/embeddings'

for model_name in models_to_run:
    print(f"\n Running model: {model_name}")

    jina_flag = model_name.startswith("jinaai/")
    generator = WMT24PPEmbeddingGenerator(
      model_name=model_name,
      csv_path=csv_path,
      jina_v3=False,
      trust_remote_code=True
    )
    output_file = generator.generate(output_dir=output_dir, use_for_bouquet=True)

    print(f"Saved to: {output_file}")

    del generator
    torch.cuda.empty_cache()
    gc.collect()

# Flores & BOUQUET


## SETUP

In [ ]:
class ModelConfig:
    """Model-specific encoding configurations."""

    KNOWN_CONFIGS = {
        'intfloat/multilingual-e5-base': {'prefix': 'query: ', 'suffix': ''},
        'intfloat/multilingual-e5-large': {'prefix': 'query: ', 'suffix': ''},
        'intfloat/multilingual-e5-large-instruct': {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery: ', 'suffix' : ''},
        'Alibaba-NLP/gte-multilingual-base': {'prefix': '', 'suffix': ''},
        'nomic-ai/nomic-embed-text-v2-moe': {'prefix': 'search_query:', 'suffix': ''},
        'sentence-transformers/paraphrase-multilingual-mpnet-base-v2': {'prefix': '', 'suffix': ''},
        'sentence-transformers/LaBSE': {'prefix': '', 'suffix': ''},
        'BAAI/bge-m3': {'prefix': '', 'suffix': ''},
        'jinaai/jina-embeddings-v3': {'prefix': '', 'suffix': ''},
        'jinaai/jina-embeddings-v5-text-small': {'prefix': '', 'suffix': ''},
        'jinaai/jina-embeddings-v5-text-nano': {'prefix': '', 'suffix': ''},
        'google/embeddinggemma-300m' : {'prefix': 'task: sentence similarity | query:', 'suffix': ''},
        "Qwen/Qwen3-Embedding-0.6B": {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery:', 'suffix' : ''},
        "Qwen/Qwen3-Embedding-4B": {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery:', 'suffix' : ''},
        "Qwen/Qwen3-Embedding-8B": {'prefix' : 'Instruct: Represent every detail of this text to avoid matching to hard negatives. \nQuery:', 'suffix' : ''},
        'default': {'prefix': '', 'suffix': ''}
    }

    @classmethod
    def get_config(cls, model_name: str, prefix: str = None, suffix: str = None) -> Dict:
        if prefix is not None or suffix is not None:
            return {'prefix': prefix or '', 'suffix': suffix or ''}
        return cls.KNOWN_CONFIGS.get(model_name, cls.KNOWN_CONFIGS['default']).copy()

class FLORESEmbeddingGenerator:
    """Generate embeddings for FLORES-200 dataset - JSON format."""

    TRANSFORMATIONS = [
        'polarity_negation',
        'role_swap',
        'antonym_replacement',
        'hypernym_substitution'
    ]

    def __init__(self, model_name: str, csv_path: str, prefix: str = None,
                 suffix: str = None, device: str = None, batch_size: int = 128, trust_remote_code=False, jina_v3=False):
        """
        Initialize generator.

        Args:
            model_name: HuggingFace model ID
            csv_path: Path to FLORES CSV
            prefix: Optional text prefix for encoding
            suffix: Optional text suffix for encoding
            device: 'cuda' or 'cpu' (auto-detect if None)
            batch_size: Batch size for encoding
        """
        self.csv_path = Path(csv_path)
        self.batch_size = batch_size

        config = ModelConfig.get_config(model_name, prefix, suffix)
        self.prefix = config['prefix']
        self.suffix = config['suffix']
        self.model_name = model_name

        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device

        print(f"\n{'='*80}")
        print(f"Initializing FLORES-200 Embedding Generator")
        print(f"{'='*80}")
        print(f"Model: {model_name}")
        print(f"Device: {device}")
        if self.prefix or self.suffix:
            print(f"Prefix: '{self.prefix}'")
            print(f"Suffix: '{self.suffix}'")

        print(f"\nLoading model...")
        self.model = SentenceTransformer(model_name, device=device, trust_remote_code=trust_remote_code)
        print(f" Model loaded")
        if jina_v3:
            self.model[0].default_task = 'text-matching' # if you try another adapter, also add it to config
            print(f" Jina Embeddings V3 was set to the text-matching adapter")

        print(f"\nLoading CSV: {self.csv_path}")
        self.df = pd.read_csv(self.csv_path, encoding = "utf-16")
        print(f"  Loaded {len(self.df)} rows")

        self.available_languages = self._discover_languages()
        print(f"  Found {len(self.available_languages)} languages")
        print(f"     First 10: {', '.join(self.available_languages[:10])}")

    def _discover_languages(self) -> List[str]:
        """Discover language codes from sentence_XXX_XXXX columns."""
        languages = []
        for col in self.df.columns:
            if col.startswith('sentence_'):
                lang_code = col.replace('sentence_', '')
                languages.append(lang_code)
        return sorted(languages)
    def _encode(self, texts: List[str]) -> np.ndarray:
        """Encode texts with model-specific prefix/suffix."""
        if self.prefix or self.suffix:
            texts = [f"{self.prefix}{text}{self.suffix}" for text in texts]

        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        return embeddings

    def generate(self, output_dir: str, languages: Optional[List[str]] = None, use_for_bouquet=False) -> Path:
        """
        Generate embeddings in  JSON structure.

        Structure: {index: {language: {original}, foil_type: {foil}}}
        Languages and foil types are at the same level.

        Args:
            output_dir: Output directory
            languages: Specific languages (None = all)

        Returns:
            Path to output JSON file
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        model_slug = self.model_name.replace('/', '_').replace('-', '_')
        if use_for_bouquet:
          dataset_name = "bouquet"
        else:
          dataset_name = "flores200"

        output_file = output_dir / f"{dataset_name}_{model_slug}.json"
        output_file_embeddings = output_dir / f"{dataset_name}_embeddings_{model_slug}.json"

        if languages is None:
            languages = self.available_languages
            print(f"\n  Using ALL {len(languages)} languages")
        else:
            invalid = set(languages) - set(self.available_languages)
            if invalid:
                print(f"  Warning: Languages not in CSV: {invalid}")
            languages = [lang for lang in languages if lang in self.available_languages]
            print(f"\n  Using {len(languages)} specified languages")

        if not languages:
            raise ValueError("No valid languages found!")

        print(f"\n{'='*80}")
        print(f"Generating Embeddings")
        print(f"{'='*80}")
        print(f"Output: {output_file}")
        print(f"Languages: {len(languages)}")

        data = {}

        print(f"\n1. Collecting successful foils...")
        successful_indices_per_foil = {}
        for foil_type in self.TRANSFORMATIONS:
            status_col = f'foil_{foil_type}_status'
            foil_col = f'foil_{foil_type}_eng_Latn'  # Note: _eng_Latn suffix!

            if status_col in self.df.columns and foil_col in self.df.columns:
                mask = (self.df[status_col] == 'success') & self.df[foil_col].notna()
                successful_indices_per_foil[foil_type] = set(self.df[mask].index.tolist())
                print(f"   {foil_type:30s}: {len(successful_indices_per_foil[foil_type]):4d} successful")

        all_successful_indices = set()
        for indices in successful_indices_per_foil.values():
            all_successful_indices.update(indices)
        all_successful_indices = sorted(all_successful_indices)

        print(f"\n   Total rows with successful foils: {len(all_successful_indices)}")

        for idx in all_successful_indices:
            data[str(idx)] = {}

        # bouquet only: carry the sentence/paragraph level flag per index
        if use_for_bouquet and 'level' in self.df.columns:
            for idx in all_successful_indices:
                data[str(idx)]['level'] = self.df.loc[idx, 'level']

        print(f"\n2. Encoding originals...")
        for language in tqdm(languages, desc="Languages"):
            sentence_col = f'sentence_{language}'

            if sentence_col not in self.df.columns:
                continue

            valid_rows = self.df.loc[all_successful_indices]
            valid_rows = valid_rows[valid_rows[sentence_col].notna()]

            if len(valid_rows) == 0:
                continue

            texts = valid_rows[sentence_col].tolist()
            indices = valid_rows.index.tolist()

            embeddings = self._encode(texts)

            for idx, text, emb in zip(indices, texts, embeddings):
                data[str(idx)][language] = {
                    'original_text': text,
                    'embedding': emb.tolist()
                }

        print(f"   Encoded originals for {len(languages)} languages")

        print(f"\n3. Encoding foils...")
        for foil_type in tqdm(self.TRANSFORMATIONS, desc="Foil types"):
            foil_col = f'foil_{foil_type}_eng_Latn'

            if foil_col not in self.df.columns:
                continue

            indices = sorted(successful_indices_per_foil[foil_type])
            valid_rows = self.df.loc[indices]
            valid_rows = valid_rows[valid_rows[foil_col].notna()]

            if len(valid_rows) == 0:
                continue

            texts = valid_rows[foil_col].tolist()
            indices = valid_rows.index.tolist()

            embeddings = self._encode(texts)

            for idx, text, emb in zip(indices, texts, embeddings):
                idx_str = str(idx)
                if idx_str in data:
                    data[idx_str][foil_type] = {
                        'text': text,
                        'embedding': emb.tolist()
                    }

        print(f" Encoded foils")

        result = {
            'metadata': {
                'model': self.model_name,
                'languages': languages,
                'foil_types': self.TRANSFORMATIONS,
                'total_indices': len(data),
                'encoding_config': {
                    'prefix': self.prefix,
                    'suffix': self.suffix,
                    'device': self.device,
                    'batch_size': self.batch_size
                }
            },
            'data': data
        }

        print(f"\n4. Writing to JSON...")
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)

        file_size_mb = output_file.stat().st_size / 1024 / 1024

        print(f"\n{'='*80}")
        print(f" Generated embeddings for {len(data)} indices")
        print(f" Saved to: {output_file}")
        print(f" Size: {file_size_mb:.1f} MB")
        print(f"{'='*80}")

        # print(f"\n Sample structure:")
        # sample_idx = str(all_successful_indices[0])
        # print(f"\ndata['{sample_idx}'] keys:")
        # sample_keys = list(data[sample_idx].keys())
        # lang_keys = [k for k in sample_keys if k not in self.TRANSFORMATIONS]
        # foil_keys = [k for k in sample_keys if k in self.TRANSFORMATIONS]
        # print(f"  Languages ({len(lang_keys)}): {lang_keys[:5]}{'...' if len(lang_keys) > 5 else ''}")
        # print(f"  Foils ({len(foil_keys)}): {foil_keys}")

        return output_file

## RUN FLORES

In [ ]:
# models_to_run = [model_to_run]
csv_path = "datasets/alee_f200.csv"
# g_drive_csv_path = '/content/AMR/flores_with_foils/flores200_ALL_successful.csv'
# g_drive output_dir = '/content/AMR/embeddings'
output_dir = 'intermediate_outputs/embeddings'

In [ ]:
models_to_run = [
    # "intfloat/multilingual-e5-base",
    # "intfloat/multilingual-e5-large",
    # "intfloat/multilingual-e5-large-instruct",
    # "Qwen/Qwen3-Embedding-0.6B",
    # "Qwen/Qwen3-Embedding-4B",
    # "Qwen/Qwen3-Embedding-8B",
    # "Alibaba-NLP/gte-multilingual-base",
    # "nomic-ai/nomic-embed-text-v2-moe",
    # "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    # "sentence-transformers/LaBSE",
    # "BAAI/bge-m3",
    # "jinaai/jina-embeddings-v3",
    # "jinaai/jina-embeddings-v5-text-small",
    # "jinaai/jina-embeddings-v5-text-nano",
    # "google/embeddinggemma-300m",
    # "ibm-granite/granite-embedding-107m-multilingual",
    # "ibm-granite/granite-embedding-278m-multilingual"
]
# models_to_run = [model_to_run]

# csv_path = '/content/AMR/flores_with_foils/flores200_ALL_successful.csv'
# output_dir = '/content/AMR/embeddings'

# generator = FLORESEmbeddingGenerator(
#     model_name=model_name,
#     csv_path=csv_path,
#     jina_v3=False,
#     trust_remote_code=True
# )

# output_file = generator.generate(output_dir=output_dir)

## RUN BOUQUET

In [ ]:
models_to_run = [
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    # "intfloat/multilingual-e5-large-instruct",
    # "Qwen/Qwen3-Embedding-0.6B",
    # "Qwen/Qwen3-Embedding-4B",
    # "Qwen/Qwen3-Embedding-8B",
    # "Alibaba-NLP/gte-multilingual-base",
    # "nomic-ai/nomic-embed-text-v2-moe",
    # "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    # "sentence-transformers/LaBSE",
    # "BAAI/bge-m3",
    # "jinaai/jina-embeddings-v3",
    # "jinaai/jina-embeddings-v5-text-small",
    # "jinaai/jina-embeddings-v5-text-nano",
    # "google/embeddinggemma-300m",
    # "ibm-granite/granite-embedding-107m-multilingual",
    # "ibm-granite/granite-embedding-278m-multilingual"
]
# models_to_run = [model_to_run]
csv_path = "datasets/alee_bq275.csv"
# g_drive_csv_path = '/content/AMR/bouquet_with_foils/bouquet_ALL_successful.csv'
# g_drive output_dir = '/content/AMR/embeddings'
output_dir = 'intermediate_outputs/embeddings'

In [ ]:
for model_name in models_to_run:
    print(f"\n Running model: {model_name}")

    jina_flag = model_name.startswith("jinaai/")

    generator = FLORESEmbeddingGenerator(
        model_name=model_name,
        csv_path=csv_path,
        jina_v3=jina_flag,
        trust_remote_code=True
    )

    output_file = generator.generate(output_dir=output_dir, use_for_bouquet=True)

    print(f"Saved to: {output_file}")

    del generator
    torch.cuda.empty_cache()
    gc.collect()


🚀 Running model: Qwen/Qwen3-Embedding-4B

Initializing FLORES-200 Embedding Generator
Model: Qwen/Qwen3-Embedding-4B
Device: cuda
Prefix: 'Instruct: Represent every detail of this text to avoid matching to hard negatives. 
Query:'
Suffix: ''

Loading model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

 Model loaded

Loading CSV: /content/AMR/bouquet_with_foils/bouquet_ALL_successful.csv
  Loaded 864 rows
  Found 275 languages
     First 10: aar_Latn, abl_Latn, afr_Latn, agr_Latn, aiq_Arab, als_Latn, amh_Ethi, ami_Latn, ane_Latn, apc_Arab

  Using ALL 275 languages

Generating Embeddings
Output: /content/AMR/embeddings/bouquet_Qwen_Qwen3_Embedding_8B.json
Languages: 275

1. Collecting successful foils...
   polarity_negation             :  669 successful
   role_swap                     :  333 successful
   antonym_replacement           :  597 successful
   hypernym_substitution         :  341 successful

   Total rows with successful foils: 864

2. Encoding originals...


Languages:   0%|          | 0/275 [00:00<?, ?it/s]

   Encoded originals for 275 languages

3. Encoding foils...


Foil types:   0%|          | 0/4 [00:00<?, ?it/s]

 Encoded foils

4. Writing to JSON...

 Generated embeddings for 864 indices
 Saved to: /content/AMR/embeddings/bouquet_Qwen_Qwen3_Embedding_8B.json
 Size: 26249.0 MB
✅ Saved to: /content/AMR/embeddings/bouquet_Qwen_Qwen3_Embedding_8B.json
